# 01 - Exploratory Data Analysis: Tabular Datasets

This notebook performs exploratory data analysis on six tabular datasets
from the Interior Minister (Ministerio del Interior) of Uruguay.

**Datasets covered:**
- `delitos_denunciados` - Reported crimes
- `violencia_domestica` - Domestic violence incidents
- `delitos_sexuales` - Sexual offenses
- `homicidios_mujeres` - Homicides of women
- `medidas_alternativas` - Alternative measures
- `sistema_carcelario` - Prison system

**Tools:** Polars for data wrangling, DuckDB for cross-dataset queries, Plotly for visualization.

In [ ]:
!pip install -q polars pyarrow duckdb plotly

In [ ]:
import polars as pl
import duckdb
import plotly.express as px
import plotly.graph_objects as go
from pathlib import Path

pl.Config.set_tbl_rows(20)
pl.Config.set_fmt_str_lengths(80)

In [ ]:
# --- Option A: Mount Google Drive ---
# from google.colab import drive
# drive.mount('/content/drive')
# DATA_DIR = Path('/content/drive/MyDrive/interior-minister/data')

# --- Option B: Download from Google Cloud Storage ---
!gcloud auth login

GCS_BUCKET = "gs://interior-minister-data/tabular"
DATA_DIR = Path("/content/data/tabular")
DATA_DIR.mkdir(parents=True, exist_ok=True)

TABULAR_FILES = [
    "delitos_denunciados.parquet",
    "violencia_domestica.parquet",
    "delitos_sexuales.parquet",
    "homicidios_mujeres.parquet",
    "medidas_alternativas.parquet",
    "sistema_carcelario.parquet",
]

for f in TABULAR_FILES:
    !gsutil cp {GCS_BUCKET}/{f} {DATA_DIR}/{f}

print("Downloaded files:")
for p in sorted(DATA_DIR.glob("*.parquet")):
    print(f"  {p.name}  ({p.stat().st_size / 1024:.1f} KB)")

In [ ]:
# --- Load delitos_denunciados ---
df_delitos = pl.read_parquet(DATA_DIR / "delitos_denunciados.parquet")

print(f"Shape: {df_delitos.shape}")
print(f"\nSchema:")
for name, dtype in df_delitos.schema.items():
    print(f"  {name}: {dtype}")

print(f"\nNull counts:")
print(df_delitos.null_count())

print(f"\nDescriptive statistics:")
df_delitos.describe()

In [ ]:
# --- Crime type distribution bar chart ---
# Adjust column names to match your actual schema
CRIME_TYPE_COL = "tipo_delito"  # update if different
COUNT_COL = "cantidad"  # update if different

crime_dist = (
    df_delitos
    .group_by(CRIME_TYPE_COL)
    .agg(pl.col(COUNT_COL).sum().alias("total"))
    .sort("total", descending=True)
    .head(20)
)

fig = px.bar(
    crime_dist.to_pandas(),
    x=CRIME_TYPE_COL,
    y="total",
    title="Top 20 Crime Types by Total Reports",
    labels={CRIME_TYPE_COL: "Crime Type", "total": "Total Reports"},
    color="total",
    color_continuous_scale="Reds",
)
fig.update_layout(xaxis_tickangle=-45, height=500)
fig.show()

In [ ]:
# --- Time series of crime counts by year ---
YEAR_COL = "anio"  # update if different

yearly = (
    df_delitos
    .group_by(YEAR_COL)
    .agg(pl.col(COUNT_COL).sum().alias("total"))
    .sort(YEAR_COL)
)

fig = px.line(
    yearly.to_pandas(),
    x=YEAR_COL,
    y="total",
    title="Reported Crimes Over Time",
    labels={YEAR_COL: "Year", "total": "Total Reports"},
    markers=True,
)
fig.update_layout(height=400)
fig.show()

In [ ]:
# --- Department-level heatmap ---
DEPT_COL = "departamento"  # update if different

heatmap_data = (
    df_delitos
    .group_by([DEPT_COL, YEAR_COL])
    .agg(pl.col(COUNT_COL).sum().alias("total"))
    .sort([DEPT_COL, YEAR_COL])
    .pivot(on=YEAR_COL, index=DEPT_COL, values="total")
    .fill_null(0)
)

# Convert to pandas for plotly heatmap
hm_pd = heatmap_data.to_pandas().set_index(DEPT_COL)
hm_pd = hm_pd.reindex(sorted(hm_pd.columns, key=int), axis=1)

fig = px.imshow(
    hm_pd,
    title="Crime Reports by Department and Year",
    labels={"x": "Year", "y": "Department", "color": "Reports"},
    color_continuous_scale="YlOrRd",
    aspect="auto",
)
fig.update_layout(height=600)
fig.show()

In [ ]:
# --- Load and profile: violencia_domestica, delitos_sexuales, homicidios_mujeres ---
datasets_group1 = {
    "violencia_domestica": "violencia_domestica.parquet",
    "delitos_sexuales": "delitos_sexuales.parquet",
    "homicidios_mujeres": "homicidios_mujeres.parquet",
}

dfs = {}
for name, filename in datasets_group1.items():
    path = DATA_DIR / filename
    df = pl.read_parquet(path)
    dfs[name] = df
    print(f"{'=' * 60}")
    print(f"Dataset: {name}")
    print(f"Shape: {df.shape}")
    print(f"Columns: {df.columns}")
    print(f"\nSchema:")
    for col_name, dtype in df.schema.items():
        print(f"  {col_name}: {dtype}")
    print(f"\nFirst 3 rows:")
    print(df.head(3))
    print(f"\nDescriptive statistics:")
    print(df.describe())
    print()

In [ ]:
# --- Load and profile: medidas_alternativas, sistema_carcelario ---
datasets_group2 = {
    "medidas_alternativas": "medidas_alternativas.parquet",
    "sistema_carcelario": "sistema_carcelario.parquet",
}

for name, filename in datasets_group2.items():
    path = DATA_DIR / filename
    df = pl.read_parquet(path)
    dfs[name] = df
    print(f"{'=' * 60}")
    print(f"Dataset: {name}")
    print(f"Shape: {df.shape}")
    print(f"Columns: {df.columns}")
    print(f"\nSchema:")
    for col_name, dtype in df.schema.items():
        print(f"  {col_name}: {dtype}")
    print(f"\nFirst 3 rows:")
    print(df.head(3))
    print(f"\nDescriptive statistics:")
    print(df.describe())
    print()

In [ ]:
# --- Cross-dataset summary table using DuckDB ---
con = duckdb.connect()

# Register all dataframes as DuckDB tables
for name, df in dfs.items():
    pdf = df.to_pandas()
    con.register(name, pdf)

# Also register delitos_denunciados
con.register("delitos_denunciados", df_delitos.to_pandas())

# Summary: row counts and column counts per dataset
summary_query = """
SELECT 'delitos_denunciados' AS dataset, COUNT(*) AS row_count FROM delitos_denunciados
UNION ALL
SELECT 'violencia_domestica', COUNT(*) FROM violencia_domestica
UNION ALL
SELECT 'delitos_sexuales', COUNT(*) FROM delitos_sexuales
UNION ALL
SELECT 'homicidios_mujeres', COUNT(*) FROM homicidios_mujeres
UNION ALL
SELECT 'medidas_alternativas', COUNT(*) FROM medidas_alternativas
UNION ALL
SELECT 'sistema_carcelario', COUNT(*) FROM sistema_carcelario
ORDER BY row_count DESC
"""

summary_df = con.execute(summary_query).fetchdf()
print("Cross-Dataset Summary:")
print(summary_df.to_string(index=False))

# Visualize row counts
fig = px.bar(
    summary_df,
    x="dataset",
    y="row_count",
    title="Row Counts Across All Tabular Datasets",
    labels={"dataset": "Dataset", "row_count": "Number of Rows"},
    color="row_count",
    color_continuous_scale="Blues",
)
fig.update_layout(xaxis_tickangle=-30, height=400)
fig.show()

con.close()

## Findings Summary

### Key Observations

1. **Data Coverage:** The six datasets span different dimensions of public safety
   in Uruguay -- from reported crimes and domestic violence to the prison system.

2. **Crime Distribution:** The bar chart reveals which crime types dominate
   reported incidents. Property crimes (hurto, rapiña) typically lead.

3. **Temporal Trends:** The time series shows year-over-year patterns in crime
   reporting, highlighting any inflection points or policy impacts.

4. **Geographic Patterns:** The department heatmap reveals spatial concentration
   of crime -- Montevideo and Canelones typically show highest volumes.

5. **Cross-Dataset Scale:** The DuckDB summary provides a quick comparison of
   dataset sizes to understand relative data availability.

### Next Steps

- Join tabular data with geographic layers (see notebook 02)
- Explore the knowledge graph for entity relationships (see notebook 03)
- Build normalized per-capita metrics using population data